# Notebook 12: Capstone — Scaling Laws and Tradeoffs

## Why This Matters

This capstone integrates everything from the series: GPU memory hierarchies, collective communications, DDP/FSDP, tensor and pipeline parallelism, gradient checkpointing, and profiling. The goal is to build a mental decision framework: given a model size, compute budget, and hardware configuration, what is the optimal training strategy? This is the question asked in every LLM infrastructure interview and every production training job design review.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass
from typing import Optional

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. The Scaling Laws Framework

Three fundamental scaling laws govern LLM training (Kaplan et al. 2020, Hoffmann et al. 2022):

**Loss as a function of compute**:
$$L(C) = \left(\frac{C_0}{C}\right)^{\alpha_C}$$

**Optimal allocation** (Chinchilla, 2022):
$$N^* = G \left(\frac{C}{6}\right)^a, \quad D^* = G^{-1} \left(\frac{C}{6}\right)^b$$

where a ≈ 0.5, b ≈ 0.5 (roughly equal scaling of parameters and tokens).

**Key practical takeaway**: for a fixed compute budget, the optimal choice is to scale parameters and training tokens **equally**. A model trained on 20x tokens per parameter outperforms one with 20x more parameters trained on the same compute budget.

In [ ]:
# Scaling law visualization

def chinchilla_loss(n_params, n_tokens, 
                    A=406.4, B=410.7, alpha=0.34, beta=0.28, E=1.69):
    """
    Chinchilla parametric loss estimate.
    L(N, D) = E + A/N^alpha + B/D^beta
    Parameters from Chinchilla paper (Table A9).
    """
    return E + A / (n_params ** alpha) + B / (n_tokens ** beta)

# Compute budget: how to allocate between N and D?
n_params_range = np.logspace(8, 12, 200)  # 100M to 1T params

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Loss vs params for fixed compute budget
ax = axes[0]
compute_budgets = [1e20, 1e21, 1e22, 1e23]  # FLOPs
for C in compute_budgets:
    losses = []
    for N in n_params_range:
        D = C / (6 * N)  # optimal D for this N and C (Chinchilla)
        if D < 1e6:
            losses.append(np.nan)
        else:
            losses.append(chinchilla_loss(N, D))
    ax.semilogx(n_params_range, losses, label=f'C={C:.0e} FLOPs')

ax.set_xlabel('Model parameters N')
ax.set_ylabel('Estimated loss')
ax.set_title('Loss vs Model Size\n(Chinchilla parametric, optimal D)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(1.5, 4.0)

# Plot 2: Optimal N and D vs compute
ax = axes[1]
C_range = np.logspace(18, 25, 100)
n_opts = []
d_opts = []
for C in C_range:
    # Rough optimal: N* = D* = sqrt(C/6)
    n_opt = (C / 6) ** 0.5
    d_opt = C / (6 * n_opt)
    n_opts.append(n_opt)
    d_opts.append(d_opt)

ax.loglog(C_range, n_opts, 'b-', label='Optimal N*', linewidth=2)
ax.loglog(C_range, d_opts, 'r--', label='Optimal D*', linewidth=2)

# Annotate notable models
models = {
    'GPT-3': (175e9 * 300e9 * 6, 175e9, 300e9),
    'Chinchilla': (70e9 * 1.4e12 * 6, 70e9, 1.4e12),
    'LLaMA-2 7B': (7e9 * 2e12 * 6, 7e9, 2e12),
}
for name, (C, N, D) in models.items():
    ax.scatter(C, N, marker='s', s=80, zorder=5)
    ax.annotate(name, xy=(C, N), fontsize=7, xytext=(C*1.5, N*1.5))

ax.set_xlabel('Compute budget (FLOPs)')
ax.set_ylabel('Optimal N / D')
ax.set_title('Chinchilla Optimal Allocation')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Token-to-parameter ratio
ax = axes[2]
n_params_b = [1, 3, 7, 13, 34, 70, 175]
optimal_tokens_T = [(n * 1e9) for n in n_params_b]  # D = N (Chinchilla)
optimal_tokens_T = [n * 20 / 1e12 for n in n_params_b]  # ~20 tokens/param rule of thumb

ax.plot(n_params_b, optimal_tokens_T, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('Model parameters (B)')
ax.set_ylabel('Optimal training tokens (T)')
ax.set_title('Chinchilla: ~20 tokens per parameter\n(approximate rule)')
ax.grid(True, alpha=0.3)

for n, t in zip(n_params_b, optimal_tokens_T):
    ax.annotate(f'{n}B params\n→ {t:.0f}T tokens', 
               xy=(n, t), fontsize=7, ha='left')

plt.tight_layout()
plt.savefig('scaling_laws_summary.png', dpi=100)
plt.show()

## 2. The Hardware Configuration Decision Tree

Given: model size, hardware, compute budget → optimal parallelism strategy.

**Decision framework**:

```
Model fits on 1 GPU?
  YES → Single GPU. Done.
  NO → How many GPUs per node? Is NVLink available?
    
    Model fits per node (FSDP)? 
      YES → FSDP with TP=1 or TP=2-8 (NVLink).
             DP across nodes for throughput.
      NO → Need PP across nodes.
           Use 3D: TP (within node) × PP (across nodes) × DP (replicas)
```

**Memory budget allocation** (rule of thumb for a training run):
- Parameters: model_params * dtype_bytes / N_gpus (FSDP)
- Gradients: same as parameters (ZeRO-2/3)
- Optimizer states: 2x parameters (Adam fp32 moments) / N_gpus
- Activations: batch * seq * d_model * n_layers (reduced by checkpointing)
- Leave 10-20% for framework overhead

In [ ]:
# Hardware configuration recommender

@dataclass
class ModelConfig:
    n_params: float    # total parameters
    n_layers: int
    d_model: int
    d_ff: int
    n_heads: int
    seq_len: int
    dtype_bytes: int = 2  # bfloat16

@dataclass  
class HardwareConfig:
    gpu_memory_gb: float
    n_gpus_per_node: int
    n_nodes: int
    nvlink_bw_gbs: float = 600.0
    ib_bw_gbs: float = 200.0
    peak_tflops: float = 77.6  # A100 dense BF16

def recommend_strategy(model: ModelConfig, hw: HardwareConfig, batch_size: int):
    total_gpus = hw.n_gpus_per_node * hw.n_nodes
    gpu_mem_bytes = hw.gpu_memory_gb * 1e9
    
    # Memory requirements
    param_bytes = model.n_params * model.dtype_bytes
    optimizer_bytes = model.n_params * 12  # Adam fp32
    
    # Can it fit on 1 GPU?
    single_gpu_mem = param_bytes + optimizer_bytes + param_bytes  # params + optimizer + gradients
    
    recommendations = {}
    
    # Try different TP degrees
    for tp in [1, 2, 4, 8]:
        if tp > hw.n_gpus_per_node:
            continue
        dp = total_gpus // tp  # no PP for now
        
        mem_per_gpu = (param_bytes + optimizer_bytes + param_bytes) / dp / tp
        # Rough activation memory
        act_mem = batch_size * model.seq_len * model.d_model * model.dtype_bytes * model.n_layers
        act_mem_per_gpu = act_mem / dp  # activations not sharded in FSDP
        act_mem_with_ckpt = act_mem_per_gpu / model.n_layers  # full checkpointing
        
        total_mem_no_ckpt = (mem_per_gpu + act_mem_per_gpu) / 1e9
        total_mem_with_ckpt = (mem_per_gpu + act_mem_with_ckpt) / 1e9
        
        recommendations[f"FSDP TP={tp} DP={dp}"] = {
            'tp': tp, 'pp': 1, 'dp': dp,
            'mem_no_ckpt_gb': total_mem_no_ckpt,
            'mem_with_ckpt_gb': total_mem_with_ckpt,
            'fits_no_ckpt': total_mem_no_ckpt < hw.gpu_memory_gb,
            'fits_with_ckpt': total_mem_with_ckpt < hw.gpu_memory_gb,
        }
    
    return recommendations

# 7B model, 32 x A100 80GB (4 nodes of 8)
llama7b = ModelConfig(7e9, 32, 4096, 11008, 32, 2048)
hw_32xa100 = HardwareConfig(80, 8, 4, 600, 200, 77.6)

print("7B model, 32 x A100 80GB (4 nodes of 8):")
print()
recs = recommend_strategy(llama7b, hw_32xa100, batch_size=4)
print(f"{'Strategy':<25} {'Mem no-ckpt (GB)':<20} {'Mem with-ckpt (GB)':<22} {'Fits?'}")
print("-" * 75)
for name, r in recs.items():
    fits = "YES (ckpt)" if r['fits_with_ckpt'] and not r['fits_no_ckpt'] else (
           "YES" if r['fits_no_ckpt'] else "NO")
    print(f"{name:<25} {r['mem_no_ckpt_gb']:<20.1f} {r['mem_with_ckpt_gb']:<22.1f} {fits}")

print()
# 70B model
llama70b = ModelConfig(70e9, 80, 8192, 28672, 64, 4096)
print("70B model, 64 x A100 80GB (8 nodes of 8):")
hw_64xa100 = HardwareConfig(80, 8, 8, 600, 200, 77.6)
recs70b = recommend_strategy(llama70b, hw_64xa100, batch_size=2)
print(f"{'Strategy':<25} {'Mem no-ckpt (GB)':<20} {'Mem with-ckpt (GB)':<22} {'Fits?'}")
print("-" * 75)
for name, r in recs70b.items():
    fits = "YES (ckpt)" if r['fits_with_ckpt'] and not r['fits_no_ckpt'] else (
           "YES" if r['fits_no_ckpt'] else "NO")
    print(f"{name:<25} {r['mem_no_ckpt_gb']:<20.1f} {r['mem_with_ckpt_gb']:<22.1f} {fits}")

## 3. The Cost-Efficiency Frontier

Training cost has two components:
1. **Compute cost**: GPU-hours × $/GPU-hour
2. **Time cost**: wall-clock time (affects iteration speed)

The optimal strategy maximizes **tokens/dollar** while meeting the time constraint.

Key tradeoffs:
- **Larger batch → better GPU utilization** but diminishing returns beyond a threshold
- **More GPUs → faster training** but communication overhead reduces efficiency
- **Longer sequences → more FLOPs per token** but activation memory grows as O(seq)
- **Mixed precision (BF16)** → 2x throughput vs FP32, at the cost of potential instability

In [ ]:
# Cost efficiency analysis

def training_cost_analysis(
    n_params, n_tokens,
    peak_tflops=77.6,   # A100 dense BF16
    mfu=0.40,            # achieved MFU
    gpu_cost_per_hour=2.00,  # $/GPU-hour (A100 spot)
    n_gpus=1,
):
    """Estimate training cost and time."""
    # Total FLOPs
    total_flops = 6 * n_params * n_tokens
    
    # Achieved throughput
    achieved_tflops = peak_tflops * mfu
    total_gpu_seconds = total_flops / (achieved_tflops * 1e12)
    
    # Wall clock time with N GPUs
    wall_clock_seconds = total_gpu_seconds / n_gpus
    wall_clock_days = wall_clock_seconds / 86400
    
    # Cost
    total_gpu_hours = total_gpu_seconds / 3600
    total_cost = total_gpu_hours * gpu_cost_per_hour
    
    return {
        'total_flops': total_flops,
        'wall_clock_days': wall_clock_days,
        'total_gpu_hours': total_gpu_hours,
        'total_cost_usd': total_cost,
        'tokens_per_dollar': n_tokens / total_cost,
    }

# Analysis for different model/token combinations with same compute budget
C = 6e23  # fixed compute budget (like Chinchilla 70B)

configs = [
    ("1B params, 100T tokens (over-trained)", 1e9, 1e14),
    ("7B params, 1T tokens (under-trained)", 7e9, 1e12),
    ("7B params, 2T tokens (standard)", 7e9, 2e12),
    ("13B params, 770B tokens (Chinchilla-ish)", 13e9, 770e9),
    ("34B params, 300B tokens", 34e9, 300e9),
    ("70B params, 140B tokens (Chinchilla)", 70e9, 140e9),
]

print(f"Training cost analysis (MFU=40%, A100 @ $2/GPU-hr, 8 GPUs):")
print()
print(f"{'Config':<45} {'Wall (days)':<14} {'Cost ($)':<12} {'Tok/$'}")
print("-" * 85)

for name, N, D in configs:
    r = training_cost_analysis(N, D, n_gpus=8)
    print(f"{name:<45} {r['wall_clock_days']:<14.1f} ${r['total_cost_usd']:<11,.0f} {r['tokens_per_dollar']:.0f}")

print()
# Show MFU impact
print("Impact of MFU on cost (70B model, 140B tokens, 64 GPUs):")
print(f"{'MFU':<10} {'Wall (days)':<14} {'Cost ($)'}")
print("-" * 40)
for mfu in [0.20, 0.30, 0.40, 0.50, 0.60]:
    r = training_cost_analysis(70e9, 140e9, mfu=mfu, n_gpus=64)
    print(f"{mfu:<10.0%} {r['wall_clock_days']:<14.1f} ${r['total_cost_usd']:,.0f}")

## 4. Interview Decision Framework

When asked "how would you train a 70B model on 64 A100s?" in an interview:

**Step 1: Memory analysis**
- 70B * 2 bytes (BF16) = 140 GB params → doesn't fit on 1 GPU (80 GB)
- With FSDP over 64 GPUs: 140 GB / 64 = 2.2 GB params per GPU
- Optimizer (Adam FP32): 70B * 12 / 64 = 13 GB per GPU
- Total model memory: ~15 GB → fits with room for activations

**Step 2: Parallelism strategy**
- Within-node (8 GPUs, NVLink): TP=8 or FSDP within node
- Across nodes (8 nodes): Pipeline (PP=8) or pure FSDP
- Recommendation: FSDP with `HYBRID_SHARD` (shard within node, replicate across) + DP=8

**Step 3: Batch size and micro-batches**
- Global batch: aim for ~1-4M tokens/step (standard for LLM pretraining)
- With seq=4096: 1M tokens / 4096 = 256 sequences
- Per GPU: 256 / 64 = 4 sequences
- With gradient accumulation: accumulate 4-8 steps if per-GPU batch too small

**Step 4: Optimizations**
- FlashAttention: mandatory for long sequences (reduces seq²  memory)
- Gradient checkpointing: full or selective depending on memory budget
- Mixed precision: BF16 params + FP32 master weights (for stability)
- Fused kernels: apex or torch.compile for LayerNorm, optimizer

In [ ]:
# Final capstone: complete training configuration recommender

def full_training_config(
    n_params_B: float,
    n_tokens_B: float,
    n_gpus: int,
    gpus_per_node: int = 8,
    gpu_memory_gb: float = 80.0,
    seq_len: int = 4096,
    mfu_target: float = 0.45,
):
    """
    Generate a complete training configuration recommendation.
    """
    n_params = n_params_B * 1e9
    n_tokens = n_tokens_B * 1e9
    n_nodes = n_gpus // gpus_per_node
    
    print(f"=" * 60)
    print(f"TRAINING CONFIGURATION RECOMMENDATION")
    print(f"=" * 60)
    print(f"Model: {n_params_B}B parameters")
    print(f"Training tokens: {n_tokens_B}B")
    print(f"Hardware: {n_gpus} x A100 {gpu_memory_gb}GB ({n_nodes} nodes of {gpus_per_node})")
    print()
    
    # --- Parallelism ---
    param_bytes = n_params * 2  # BF16
    optim_bytes = n_params * 12  # Adam FP32
    
    # FSDP across all GPUs
    mem_fsdp = (param_bytes + optim_bytes + param_bytes) / n_gpus / 1e9
    
    # Recommended TP
    tp = min(gpus_per_node, max(1, int(np.ceil(param_bytes / (gpu_memory_gb * 0.5 * 1e9 * gpus_per_node)))))
    tp = min(tp, 8)
    dp = n_gpus // tp
    
    print(f"PARALLELISM:")
    print(f"  FSDP memory per GPU (no activations): {mem_fsdp:.1f} GB")
    if mem_fsdp < gpu_memory_gb * 0.5:
        print(f"  Strategy: FSDP FULL_SHARD, TP=1 (model fits easily)")
    elif mem_fsdp < gpu_memory_gb * 0.7:
        print(f"  Strategy: FSDP HYBRID_SHARD (shard within node, replicate across)")
    else:
        print(f"  Strategy: TP={tp} + FSDP or 3D (TP+PP+DP)")
    print()
    
    # --- Batch size ---
    tokens_per_step = 1e6  # 1M tokens/step (standard LLM pretraining)
    global_batch = int(tokens_per_step / seq_len)
    batch_per_gpu = global_batch // n_gpus
    grad_accum = max(1, 4 // batch_per_gpu)  # target at least 4 per GPU
    effective_batch_per_gpu = batch_per_gpu * grad_accum
    
    print(f"BATCH SIZE:")
    print(f"  Target: {tokens_per_step/1e6:.0f}M tokens/step")
    print(f"  Global batch: {global_batch} sequences")
    print(f"  Per GPU: {batch_per_gpu} (+ grad accum x{grad_accum} = {effective_batch_per_gpu})")
    print()
    
    # --- Activation memory ---
    # Rough: batch * seq * d_model (estimate d_model ~ sqrt(n_params/4))
    d_model_est = int((n_params / 4 / 48) ** 0.5)  # rough estimate
    n_layers_est = int(n_params / (4 * d_model_est ** 2))
    act_no_ckpt = effective_batch_per_gpu * seq_len * d_model_est * 2 * n_layers_est / 1e9
    act_with_ckpt = effective_batch_per_gpu * seq_len * d_model_est * 2 / 1e9  # 1 layer
    
    total_mem_no_ckpt = mem_fsdp + act_no_ckpt
    total_mem_with_ckpt = mem_fsdp + act_with_ckpt
    
    print(f"MEMORY (estimated d_model={d_model_est}, n_layers={n_layers_est}):")
    print(f"  Model + optimizer: {mem_fsdp:.1f} GB/GPU")
    print(f"  Activations (no ckpt): {act_no_ckpt:.1f} GB/GPU -> total {total_mem_no_ckpt:.1f} GB")
    print(f"  Activations (full ckpt): {act_with_ckpt:.1f} GB/GPU -> total {total_mem_with_ckpt:.1f} GB")
    grad_ckpt = "required" if total_mem_no_ckpt > gpu_memory_gb else "optional"
    print(f"  Gradient checkpointing: {grad_ckpt}")
    print()
    
    # --- Training cost ---
    total_flops = 6 * n_params * n_tokens
    peak_tflops = 77.6
    achieved_tflops = peak_tflops * mfu_target
    wall_clock_s = total_flops / (achieved_tflops * 1e12 * n_gpus)
    wall_clock_days = wall_clock_s / 86400
    total_gpu_hours = wall_clock_s / 3600 * n_gpus
    cost_usd = total_gpu_hours * 2.00
    
    print(f"COST ESTIMATE (MFU={mfu_target:.0%}, $2/GPU-hr):")
    print(f"  Wall clock: {wall_clock_days:.1f} days")
    print(f"  GPU-hours: {total_gpu_hours:,.0f}")
    print(f"  Estimated cost: ${cost_usd:,.0f}")
    print()
    print(f"OPTIMIZATIONS TO APPLY:")
    print(f"  - FlashAttention-2 (mandatory for seq >= 2048)")
    print(f"  - Mixed precision: BF16 params, FP32 master weights")
    print(f"  - Gradient checkpointing: {'selective (attn only)' if grad_ckpt == 'optional' else 'full (every layer)'}")
    print(f"  - Fused kernels: torch.compile or apex")
    print(f"  - DDP bucket tuning: bucket_cap_mb >= 100")
    print(f"  - Profile target: MFU >= {mfu_target:.0%}")
    print("=" * 60)

# Run for common model sizes
full_training_config(7, 2000, n_gpus=8, seq_len=4096)
print()
full_training_config(70, 2000, n_gpus=64, seq_len=4096)

## 5. Series Summary: Everything You Need to Know

### The Full Picture

| Topic | Key concept | Critical formula / number |
|-------|-------------|--------------------------|
| GPU memory hierarchy | HBM is the bottleneck | A100: 2 TB/s, PCIe: 32 GB/s |
| NCCL collectives | Ring all-reduce is optimal | 2*(N-1)/N * P bytes per GPU |
| DDP | Gradient bucketing + overlap | bucket_cap_mb=25, find_unused=False |
| FSDP / ZeRO | Shard params + gradients + optimizer | Memory = 16 bytes/param / N_gpus (Adam) |
| Ray Core | Tasks + Actors + Object store | Zero-copy sharing via Plasma |
| Ray Train | DDP orchestration + fault tolerance | prepare_model + prepare_data_loader |
| Tensor parallelism | Column+row parallel fused | 1 all-reduce per layer (Megatron) |
| Pipeline parallelism | 1F1B schedule | Bubble = (N-1)/(M+N-1) |
| Megatron patterns | 3D: TP × PP × DP | Sequence parallelism for LayerNorm |
| Gradient checkpointing | Recompute activations | ~33% compute overhead |
| Profiling | MFU = achieved / peak FLOPS | Good: 40-50%+ |
| Scaling laws | Chinchilla optimal | ~20 tokens per parameter |

### Decision Rules

**Use DDP when**: model fits on 1 GPU (just need data parallelism)
**Use FSDP when**: model doesn't fit on 1 GPU but fits sharded across N GPUs
**Add TP when**: FSDP not enough; must stay within NVLink domain (≤ 8)
**Add PP when**: even TP+FSDP insufficient; model spans multiple nodes
**Always use**: FlashAttention, BF16, fused kernels, gradient checkpointing

### The Interview Answer Template

*"Given model X on hardware Y, I would:"*
1. Calculate per-GPU memory budget (params + optimizer + activations)
2. Choose minimal parallelism that fits (DDP → FSDP → FSDP+TP → 3D)
3. Set batch size to maximize MFU (target ≥ 40%)
4. Apply FlashAttention + gradient checkpointing
5. Profile to verify MFU and iterate

In [ ]:
# Final: the series in a single visualization

fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.35)

axes = [fig.add_subplot(gs[i//4, i%4]) for i in range(8)]

# 1. Memory hierarchy
ax = axes[0]
levels = ['Registers', 'L1/Shared', 'L2', 'HBM', 'PCIe', 'NVLink']
bw = [30000, 10000, 4000, 2000, 0.032, 0.6]
colors1 = ['darkblue', 'blue', 'steelblue', 'lightblue', 'orange', 'green']
ax.barh(levels, bw, color=colors1, alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('Bandwidth (TB/s)')
ax.set_title('01: GPU Memory Hierarchy')
ax.grid(True, alpha=0.3, axis='x')

# 2. Ring all-reduce cost
ax = axes[1]
n_gpus = np.arange(2, 65)
ring_cost = 2 * (n_gpus - 1) / n_gpus
ax.plot(n_gpus, ring_cost, 'b-', linewidth=2, label='Ring (constant!)')
ax.axhline(y=2.0, color='r', linestyle='--', label='PS (always 2x)')
ax.set_xlabel('Number of GPUs')
ax.set_ylabel('Relative comm cost per GPU')
ax.set_title('02-03: Ring All-Reduce Scaling')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. FSDP memory savings
ax = axes[2]
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]
mem_ddp = [84] * len(n_gpus_list)  # 7B model, doesn't change
mem_fsdp = [84 / n for n in n_gpus_list]  # scales linearly
ax.plot(n_gpus_list, mem_ddp, 'r-o', label='DDP')
ax.plot(n_gpus_list, mem_fsdp, 'b-o', label='FSDP')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='80GB limit')
ax.set_xlabel('GPUs')
ax.set_ylabel('Memory/GPU (GB)')
ax.set_title('04: FSDP Memory Scaling (7B model)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. Bubble fraction
ax = axes[3]
m_vals = np.arange(1, 33)
for n in [2, 4, 8]:
    bubble = (n - 1) / (m_vals + n - 1)
    ax.plot(m_vals, bubble, label=f'N={n} stages')
ax.set_xlabel('Micro-batches M')
ax.set_ylabel('Bubble fraction')
ax.set_title('07-08: Pipeline Bubble vs Micro-batches')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)

# 5. TP communication overhead
ax = axes[4]
tp_degrees = [1, 2, 4, 8, 16]
# Simulated comm/compute ratio
comm_ratios_nvlink = [0, 0.01, 0.03, 0.06, 0.20]
comm_ratios_ib = [0, 0.03, 0.08, 0.18, 0.50]
ax.plot(tp_degrees, comm_ratios_nvlink, 'b-o', label='NVLink (600 GB/s)')
ax.plot(tp_degrees, comm_ratios_ib, 'r-o', label='InfiniBand (200 GB/s)')
ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='10% overhead')
ax.set_xlabel('TP degree')
ax.set_ylabel('Comm/compute ratio')
ax.set_title('07: Tensor Parallel Overhead')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 6. Gradient checkpointing tradeoff
ax = axes[5]
k_vals = range(1, 25)
mem_norm = [k / 24 for k in k_vals]  # normalized memory
overhead = [0.33] * len(k_vals)  # constant overhead
ax.plot(list(k_vals), mem_norm, 'b-', linewidth=2, label='Memory (relative)')
ax.axhline(y=0.33, color='r', linestyle='--', label='~33% compute overhead')
ax.set_xlabel('Checkpoint interval k (layers)')
ax.set_ylabel('Relative value')
ax.set_title('10: Gradient Checkpointing')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 7. MFU optimization journey
ax = axes[6]
optimizations = ['Baseline', '+Bucketing', '+FlashAttn', '+Fused\nKernels', '+Tuned\nBatch']
mfu_vals = [0.20, 0.30, 0.40, 0.47, 0.52]
bars = ax.bar(range(len(optimizations)), mfu_vals, color=['red', 'orange', 'gold', 'yellowgreen', 'green'])
ax.set_xticks(range(len(optimizations)))
ax.set_xticklabels(optimizations, fontsize=7)
ax.set_ylabel('Model FLOP Utilization')
ax.set_title('11: MFU Optimization Journey')
ax.axhline(y=0.40, color='gray', linestyle='--', alpha=0.5, label='Target 40%')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')
for bar, v in zip(bars, mfu_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.0%}', ha='center', va='bottom', fontsize=8)

# 8. Scaling law
ax = axes[7]
C_range = np.logspace(20, 25, 50)
n_opts = [(c/6)**0.5 / 1e9 for c in C_range]
ax.loglog(C_range, n_opts, 'b-', linewidth=2)
notable = [('GPT-3\n175B', 175e9*300e9*6, 175), 
           ('Chinchilla\n70B', 70e9*1.4e12*6, 70),
           ('LLaMA-2\n70B', 70e9*2e12*6, 70)]
for name, C, N in notable:
    ax.scatter(C, N, s=80, zorder=5)
    ax.annotate(name, xy=(C, N), fontsize=7)
ax.set_xlabel('Compute (FLOPs)')
ax.set_ylabel('Params (B)')
ax.set_title('12: Chinchilla Optimal Scale')
ax.grid(True, alpha=0.3)

plt.suptitle('Series 39: Distributed ML & Ray — Complete Summary', 
             fontsize=14, fontweight='bold')
plt.savefig('series_summary.png', dpi=100, bbox_inches='tight')
plt.show()

print("Series 39 complete! You now have a comprehensive understanding of:")
print("  - GPU memory hierarchy and roofline model")
print("  - NCCL collective communications")
print("  - DDP, FSDP/ZeRO sharding")
print("  - Ray Core fundamentals and Ray Train")
print("  - Tensor and pipeline parallelism")
print("  - Megatron-LM 3D parallelism patterns")
print("  - Gradient checkpointing memory-compute tradeoffs")
print("  - Profiling and MFU optimization")
print("  - Scaling laws and training cost analysis")

## Summary

### Complete Series Reference

| Notebook | Core skill | Key number to remember |
|----------|-----------|----------------------|
| 01: GPU Memory | Roofline model | A100: 2 TB/s HBM, 312 TFLOP/s FP16 |
| 02: NCCL | Ring all-reduce | 2*(N-1)/N * P bytes/GPU (constant!) |
| 03: DDP | Gradient bucketing | bucket_cap_mb=25, find_unused=False |
| 04: FSDP | ZeRO stages | 16 bytes/param (Adam BF16) → 16/N_gpus |
| 05: Ray Core | Tasks + Actors | Zero-copy object store via Plasma |
| 06: Ray Train | DDP orchestration | prepare_model + train.report |
| 07: Tensor Parallel | Column+row parallel | 1 all-reduce/layer (Megatron fusion) |
| 08: Pipeline Parallel | 1F1B schedule | Bubble = (N-1)/(M+N-1) |
| 09: Megatron Patterns | 3D parallelism | TP×PP×DP, sequence parallel |
| 10: Gradient Checkpointing | Recompute activations | 33% compute overhead |
| 11: Profiling | MFU measurement | Good: 40-50%+, target: minimize waste |
| 12: Scaling Laws | Chinchilla optimal | ~20 tokens/parameter for training |

**The single most important insight from this series**: distributed training efficiency is fundamentally about keeping GPUs compute-bound (high arithmetic intensity) and communication invisible (overlapped with compute). Every technique -- from FlashAttention to 1F1B scheduling to gradient bucketing -- serves this goal.